
# CC235 - Trabajo Final: Procesamiento de Imágenes
# Clasificación de Radiografías COVID-19


## Instrucciones para usar la GPU con TensorFlow (CUDA)
Si tienes una tarjeta gráfica NVIDIA y quieres acelerar el entrenamiento con TensorFlow, sigue estos pasos antes de ejecutar el notebook:
1. **Instala los drivers NVIDIA más recientes** para tu tarjeta gráfica.
2. **Instala CUDA Toolkit** (versión recomendada para TensorFlow: 11.2 o 11.8). Descárgalo desde: https://developer.nvidia.com/cuda-toolkit
3. **Instala cuDNN** (compatible con la versión de CUDA que instalaste). Descárgalo desde: https://developer.nvidia.com/cudnn
4. **Agrega las rutas de CUDA y cuDNN** a las variables de entorno de Windows (`PATH`).
5. **Verifica la instalación** ejecutando en una celda:
```python
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
```
Si ves una lista con tu GPU, ¡ya está lista para usarse! Si está vacía, revisa los pasos anteriores.

**Nota:** TensorFlow debe estar instalado en una versión compatible con CUDA/cuDNN. Puedes consultar la compatibilidad oficial aquí: https://www.tensorflow.org/install/source_windows#gpu


In [1]:
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from skimage.feature import hog, local_binary_pattern
from skimage import io, color
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

# Configuración para usar GPU si está disponible
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs detectadas: {len(gpus)}. TensorFlow usará la(s) GPU disponible(s).")
    except RuntimeError as e:
        print(e)
else:
    print("No se detectó GPU. TensorFlow usará la CPU.")

No se detectó GPU. TensorFlow usará la CPU.



# 1. CARGA Y PREPROCESAMIENTO


In [ ]:
DATASET_PATH = "dataset"  # Carpeta con subcarpetas COVID-19, Normal, Lung_Opacity, Viral_Pneumonia
IMG_SIZE = 256

classes = ["COVID-19", "Normal", "Lung_Opacity", "Viral_Pneumonia"]

images = []
labels = []

for idx, cls in enumerate(classes):
    folder = os.path.join(DATASET_PATH, cls)
    for file in os.listdir(folder):
        img_path = os.path.join(folder, file)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)
            labels.append(idx)

images = np.array(images)
labels = np.array(labels)

# Normalización
images_norm = images / 255.0

# División en entrenamiento y validación
X_train, X_val, y_train, y_val = train_test_split(images_norm, labels, test_size=0.2, random_state=42, stratify=labels)




# 2. PIPELINE CLÁSICO


In [ ]:
def extract_features(img):
    # HOG
    hog_features = hog(img, orientations=9, pixels_per_cell=(16,16),
                       cells_per_block=(2,2), block_norm='L2-Hys')
    # LBP
    lbp = local_binary_pattern(img, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0,10))
    lbp_hist = lbp_hist / lbp_hist.sum()
    # Combinamos
    features = np.hstack([hog_features, lbp_hist])
    return features

X_train_feat = np.array([extract_features(img) for img in X_train])
X_val_feat = np.array([extract_features(img) for img in X_val])

# Clasificador SVM
svm = SVC(kernel='rbf', probability=True)
svm.fit(X_train_feat, y_train)
y_pred_svm = svm.predict(X_val_feat)

print("=== Clasificador SVM ===")
print(classification_report(y_val, y_pred_svm, target_names=classes))

# Clasificador Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_feat, y_train)
y_pred_rf = rf.predict(X_val_feat)

print("=== Clasificador Random Forest ===")
print(classification_report(y_val, y_pred_rf, target_names=classes))



# 3. MODELO DEEP LEARNING (ResNet50)

In [ ]:

# Convertimos imágenes a 3 canales
X_train_dl = np.stack([X_train]*3, axis=-1)
X_val_dl = np.stack([X_val]*3, axis=-1)

# ImageDataGenerator con Data Augmentation
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator()

train_gen = train_datagen.flow(X_train_dl, tf.keras.utils.to_categorical(y_train, num_classes=4), batch_size=32)
val_gen = val_datagen.flow(X_val_dl, tf.keras.utils.to_categorical(y_val, num_classes=4), batch_size=32)

# ResNet50 base
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(4, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# Congelamos las capas base
for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# Entrenamiento
history = model.fit(train_gen, validation_data=val_gen, epochs=10, verbose=1)

# Evaluación
val_loss, val_acc = model.evaluate(val_gen)
print(f"Accuracy ResNet50: {val_acc:.2f}")


# 4. GRÁFICA COMPARATIVA


In [ ]:

svm_acc = np.mean(y_pred_svm == y_val)
rf_acc = np.mean(y_pred_rf == y_val)

plt.bar(['SVM', 'RandomForest', 'ResNet50'], [svm_acc, rf_acc, val_acc])
plt.ylabel('Accuracy')
plt.title('Comparación de Modelos')
plt.show()